In [ ]:
import os
import cv2
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Concatenate, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


In [ ]:
# Function to split images into tiles
def split_image_into_tiles(image_path, mask_path, tile_size, size):
    img = tifffile.imread(image_path)
    mask = tifffile.imread(mask_path)
    mask = mask[:, :, 0] if len(mask.shape) == 3 else mask

    tiles_img, tiles_mask = [], []
    for x in range(0, img.shape[1], tile_size):
        for y in range(0, img.shape[0], tile_size):
            tile_img = img[y:y+tile_size, x:x+tile_size, :]
            tile_mask = mask[y:y+tile_size, x:x+tile_size]

            tile_img = cv2.resize(tile_img, (size, size))
            tile_mask = cv2.resize(tile_mask, (size, size))
            tile_mask = (tile_mask > 0).astype(np.uint8)

            tiles_img.append(tile_img)
            tiles_mask.append(tile_mask)

    return np.array(tiles_img), np.array(tiles_mask)

# Load dataset
def load_data(image_dir, mask_dir, tile_size=256, size=256):
    images, masks = [], []
    image_filenames = sorted(os.listdir(image_dir))
    mask_filenames = sorted(os.listdir(mask_dir))

    for image_filename in image_filenames:
        if image_filename.endswith(".TIF"):
            mask_filename = image_filename.replace(".TIF", "_mask.TIF")
            if mask_filename in mask_filenames:
                img_path = os.path.join(image_dir, image_filename)
                mask_path = os.path.join(mask_dir, mask_filename)
                img, mask = split_image_into_tiles(img_path, mask_path, tile_size, size)
                images.extend(img)
                masks.extend(mask)
    return np.array(images), np.array(masks)

# Paths
image_dir = "../datasets/images"
mask_dir = "../datasets/masks"
size = 256

# Load data
tiles_img, tiles_mask = load_data(image_dir, mask_dir, tile_size=size, size=size)

# Split sets
X_train, X_test, y_train, y_test = train_test_split(tiles_img, tiles_mask, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Reshape masks
y_train = y_train[..., np.newaxis]
y_val = y_val[..., np.newaxis]
y_test = y_test[..., np.newaxis]


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Concatenate
from tensorflow.keras.optimizers import Adam
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tifffile
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split



def resnet50_segmentation(input_size=(256, 256, 3), freeze_encoder=True):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_size)

    if freeze_encoder:
        for layer in base_model.layers:
            layer.trainable = False

    x = base_model.output  # output from deepest layer (8x8 if input 256x256)

    # Lightweight decoder (upsampling)
    x = Conv2DTranspose(512, (3, 3), strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(256, (3, 3), strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(128, (3, 3), strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(64, (3, 3), strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(32, (3, 3), strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    output = Conv2D(1, (1, 1), activation='sigmoid')(x)
    output = tf.image.resize(output, (input_size[0], input_size[1]), method='bilinear')

    model = Model(inputs=base_model.input, outputs=output)
    #model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'], dice_coef, iou_coef)
 
    model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef, iou_coef])


    
    return model

# Usage
size = 256
model = resnet50_segmentation(input_size=(size, size, 3), freeze_encoder=True)
model.summary()


In [ ]:
# LR schedule
def lr_schedule(epoch):
    initial_lr = 1e-4
    decay = 0.9
    return initial_lr * (decay ** (epoch // 10))

lr_scheduler = LearningRateScheduler(lr_schedule)

# Data augmentation
datagen = ImageDataGenerator(rescale=1./255,
                             shear_range=0.2,
                             zoom_range=0.2,
                             horizontal_flip=True,
                             rotation_range=20,
                             width_shift_range=0.2,
                             height_shift_range=0.2,
                             brightness_range=[0.8, 1.2])

# Callbacks
checkpointer = ModelCheckpoint("best_resnetnet.h5", monitor="val_dice_coef", mode="max",
                               save_best_only=True, verbose=1)
earlyStopping = EarlyStopping(monitor="val_dice_coef", patience=5, mode="max", verbose=1)


In [ ]:
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_val/255.0, y_val),
                    epochs=50,
                    callbacks=[lr_scheduler, earlyStopping, checkpointer])


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample



def bootstrap_confidence_interval(y_true, y_pred, metric_fn, n_bootstraps=100, alpha=0.95):
    """Bootstrap CI + std for a given metric"""
    stats = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        indices = np.random.randint(0, n, n)
        if metric_fn.__name__ == "roc_auc_score":  # ROC-AUC requires probs
            stat = metric_fn(y_true[indices], y_pred[indices])
        else:  # Binary metrics
            stat = metric_fn(y_true[indices], y_pred[indices])
        stats.append(stat)
    
    stats = np.array(stats)
    mean_val = np.mean(stats)
    std_val  = np.std(stats)
    lower = np.percentile(stats, ((1 - alpha) / 2) * 100)
    upper = np.percentile(stats, (alpha + (1 - alpha) / 2) * 100)
    
    return mean_val, std_val, (lower, upper)


# --- Evaluate model ---
loss, acc, dice, iou = model.evaluate(X_test/255.0, y_test)
print(f"Test Loss: {loss:.4f}, Accuracy: {acc:.4f}, Dice: {dice:.4f}, IoU: {iou:.4f}")

# --- Predictions ---
y_pred = model.predict(X_test/255.0)
y_pred_bin = (y_pred > 0.5).astype(np.uint8)

# Flatten
y_true_flat = y_test.flatten()
y_pred_flat = y_pred_bin.flatten()
y_pred_probs = y_pred.flatten()

# --- Metrics ---
metrics = {
    "Accuracy": lambda yt, yp: np.mean(yt == yp),
    "Precision": lambda yt, yp: precision_score(yt, yp),
    "Recall": lambda yt, yp: recall_score(yt, yp),
    "F1-score": lambda yt, yp: f1_score(yt, yp),
    "ROC-AUC": lambda yt, yp: roc_auc_score(yt, yp),
    "Dice": lambda yt, yp: (2*np.sum(yt*yp))/(np.sum(yt)+np.sum(yp)+1e-7),
    "IoU": lambda yt, yp: np.sum(yt*yp)/(np.sum(yt)+np.sum(yp)-np.sum(yt*yp)+1e-7)
}

print("\n📊 Metrics with 95% Confidence Intervals:")
for name, fn in metrics.items():
    if name == "ROC-AUC":
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_probs, fn)
    else:
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_flat, fn)
    print(f"{name}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {low:.4f} – {high:.4f})")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Flatten ground truth and predictions
y_true_flat = y_test.flatten()
y_pred_flat = (y_pred.flatten() > 0.5).astype(int)  # threshold at 0.5

# Confusion Matrix
cm = confusion_matrix(y_true_flat, y_pred_flat)
print("Confusion Matrix:\n", cm)

# Classification Report
print(classification_report(y_true_flat, y_pred_flat))

# Heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Forest","Forest"],
            yticklabels=["Non-Forest","Forest"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Plot training history for loss, accuracy, dice, iou
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('Resnet Segmentation Training History')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Epochs: 1 to 14 (early stopping at 14)
epochs = range(1, 15)

# Training metrics
train_loss = [
    0.5260, 0.3756, 0.3543, 0.3498, 0.3429, 0.3375, 0.3438, 0.3298, 0.3200, 0.3216,
    0.3353, 0.3186, 0.3115, 0.3095
]

train_accuracy = [
    0.7425, 0.8360, 0.8407, 0.8402, 0.8447, 0.8477, 0.8426, 0.8499, 0.8545, 0.8532,
    0.8443, 0.8544, 0.8562, 0.8571
]

train_dice = [
    0.6459, 0.7556, 0.7749, 0.7767, 0.7836, 0.7910, 0.7862, 0.7981, 0.7991, 0.8023,
    0.7939, 0.8027, 0.8082, 0.8123
]

train_iou = [
    0.4810, 0.6085, 0.6336, 0.6367, 0.6459, 0.6557, 0.6498, 0.6652, 0.6674, 0.6715,
    0.6610, 0.6725, 0.6801, 0.6856
]

# Validation metrics
val_loss = [
    0.6478, 0.5311, 0.4239, 0.3352, 0.3062, 0.2869, 0.2794, 0.2808, 0.2694, 0.3118,
    0.2742, 0.2717, 0.2726, 0.2722
]

val_accuracy = [
    0.8427, 0.8572, 0.8773, 0.8721, 0.8768, 0.8816, 0.8833, 0.8755, 0.8858, 0.8611,
    0.8823, 0.8822, 0.8816, 0.8842
]

val_dice = [
    0.5409, 0.6182, 0.6904, 0.7686, 0.7918, 0.8084, 0.8173, 0.8176, 0.8289, 0.7911,
    0.8235, 0.8217, 0.8225, 0.8199
]

val_iou = [
    0.3717, 0.4490, 0.5291, 0.6266, 0.6576, 0.6807, 0.6933, 0.6936, 0.7099, 0.6559,
    0.7021, 0.6995, 0.7006, 0.6968
]

In [ ]:
plt.figure(figsize=(16, 12))

# Plot 1: Loss
plt.subplot(2, 2, 1)
plt.plot(epochs, train_loss, 'bo-', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss, 'r-o', label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy
plt.subplot(2, 2, 2)
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(epochs, val_accuracy, 'r-o', label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Dice Coefficient
plt.subplot(2, 2, 3)
plt.plot(epochs, train_dice, 'bo-', label='Training Dice Coefficient', linewidth=2)
plt.plot(epochs, val_dice, 'r-o', label='Validation Dice Coefficient', linewidth=2)
plt.title('Training and Validation Dice Coefficient', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Dice Coefficient')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 4: IoU
plt.subplot(2, 2, 4)
plt.plot(epochs, train_iou, 'bo-', label='Training IoU', linewidth=2)
plt.plot(epochs, val_iou, 'r-o', label='Validation IoU', linewidth=2)
plt.title('Training and Validation IoU', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust layout and show
plt.tight_layout()
plt.show()